# 05. Model Evaluation & Comparative Benchmark
## Text Summarization Using Pre-trained Models

**Project:** Academic NLP Summarization Pipeline  
**Objective:** Perform multi-axis quantitative and qualitative evaluation comparing heuristic (Lead-3), graph-based extractive (TextRank), and pretrained sequence-to-sequence abstractive models (T5-Small and BART-Large-CNN) on the CNN/DailyMail benchmark using ROUGE-1, ROUGE-2, and ROUGE-L metrics.

In [ ]:
import sys
import json
from pathlib import Path

# Add project root to sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from IPython.display import Image, display

from config import METRICS_DIR, SUMMARIES_DIR, FIGURES_DIR
print("Evaluation modules loaded successfully.")

## 1. Quantitative Benchmark Results
We inspect the benchmark metrics calculated across 50 test samples from CNN/DailyMail.

In [ ]:
metrics_csv = METRICS_DIR / "rouge_scores.csv"
metrics_df = pd.read_csv(metrics_csv)
display(metrics_df)

## 2. Visualization of Evaluation Metrics
We inspect the 3 generated comparison figures:

In [ ]:
eval_plots = [
    "rouge_comparison.png",
    "model_latency_comparison.png",
    "generated_summary_lengths.png"
]

for plot_name in eval_plots:
    p = FIGURES_DIR / plot_name
    if p.exists():
        print(f"Figure: {plot_name}")
        display(Image(filename=str(p)))
    else:
        print(f"Figure not found: {p}")

## 3. Qualitative Sample Comparison
Examining actual generated summaries across all four architectures for a single news article:

In [ ]:
qual_json_path = SUMMARIES_DIR / "qualitative_comparison.json"
with open(qual_json_path, "r", encoding="utf-8") as f:
    qual_samples = json.load(f)

sample_0 = qual_samples[0]
print(f"=== Article ID: {sample_0['id']} ===\n")
print(f"--- Human Reference Summary ---\n{sample_0['human_reference']}\n")
print(f"--- Lead-3 Summary ---\n{sample_0['lead3_summary']}\n")
print(f"--- TextRank Summary ---\n{sample_0['textrank_summary']}\n")
print(f"--- T5-Small Summary ---\n{sample_0['t5_summary']}\n")
print(f"--- BART-Large-CNN Summary ---\n{sample_0['bart_summary']}\n")

## 4. In-Depth Comparative Discussion & Academic Insights

### Extractive vs. Abstractive Summarization Trade-offs
- **Extractive (Lead-3 & TextRank):**
  - *Strengths:* Guaranteed 100% factual fidelity (no hallucination), zero GPU requirement, sub-10ms inference.
  - *Weaknesses:* Verbose (averages 62–66 words vs. 35–43 for humans), rigid sentences that often contain superfluous context, lower ROUGE precision.
- **Abstractive (BART & T5):**
  - *Strengths:* Synthesizes and paraphrases information, produces concise summaries (41.9 words) matching human length, achieves substantially higher ROUGE scores (BART ROUGE-1 F1: 36.38% vs TextRank: 27.16%).
  - *Weaknesses:* Requires GPU memory (though optimized via FP16 to ~1.6GB VRAM), slower inference (~0.97s per article), and carries potential hallucination risks if inputs contain conflicting factual claims.